# Lab | Data Aggregation and Filtering

In this lab we continue working with the marketing customer analysis dataset. First I load the data and clean the column names so they are easier to use.

In [ ]:
import pandas as pd

url = "https://raw.githubusercontent.com/data-bootcamp-v4/data/main/marketing_customer_analysis.csv"
df = pd.read_csv(url)

df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
)

df.head()

## 1. Create a new DataFrame that only includes customers who:
- have a total claim amount below 1,000
- responded Yes to the last marketing campaign

In [ ]:
# fill missing responses before using str.lower()
response_clean = df["response"].fillna("").str.lower()

low_claim_yes_customers = df[
    (df["total_claim_amount"] < 1000) &
    (response_clean == "yes")
]

print("Number of customers:", len(low_claim_yes_customers))
low_claim_yes_customers.head()

## 2. Analyze average premium, customer lifetime value and claims by policy type and gender for customers who responded Yes.

In [ ]:
responders = df[response_clean == "yes"]

segment_analysis = (
    responders
    .groupby(["policy_type", "gender"])
    .agg(
        customer_count=("customer", "count"),
        average_monthly_premium=("monthly_premium_auto", "mean"),
        average_customer_lifetime_value=("customer_lifetime_value", "mean"),
        average_total_claim_amount=("total_claim_amount", "mean")
    )
    .round(2)
)

segment_analysis.sort_values(
    by=["average_customer_lifetime_value", "average_total_claim_amount"],
    ascending=[False, True]
)

### Conclusion
Personal Auto female customers have the highest average customer lifetime value. Corporate Auto male customers have the lowest average claim amount, which could make them a relatively low-risk segment. The customer count should also be considered because averages from small groups may be less reliable.

## 3. Count customers in each state and keep states with more than 500 customers.

In [ ]:
customers_by_state = (
    df.groupby("state")
    .agg(total_customers=("customer", "count"))
    .sort_values("total_customers", ascending=False)
)

states_over_500 = customers_by_state[
    customers_by_state["total_customers"] > 500
]

states_over_500

## 4. Find the maximum, minimum and median customer lifetime value by education and gender.

In [ ]:
# make sure CLV is numeric before calculating
df["customer_lifetime_value"] = pd.to_numeric(
    df["customer_lifetime_value"], errors="coerce"
)

clv_summary = (
    df.groupby(["education", "gender"])
    .agg(
        maximum_clv=("customer_lifetime_value", "max"),
        minimum_clv=("customer_lifetime_value", "min"),
        median_clv=("customer_lifetime_value", "median")
    )
    .round(2)
    .sort_values("median_clv", ascending=False)
)

clv_summary

### Conclusion
The highest typical CLV is for male customers with High School or Below education, with a median of about 6,286.73. The lowest is for female customers with a Doctor education, with a median of about 5,332.46. The maximum values are much higher than the medians, so a few very high-value customers probably pull up the averages.

## Bonus

## 5. Show the number of policies sold by state and month, with states as rows and months as columns.

In [ ]:
# convert the date column and extract the month name
df["effective_to_date"] = pd.to_datetime(
    df["effective_to_date"], format="%m/%d/%y"
)
df["month"] = df["effective_to_date"].dt.month_name()

policies_state_month = pd.pivot_table(
    df,
    index="state",
    columns="month",
    values="policy",
    aggfunc="count",
    fill_value=0
)

policies_state_month

## 6. Display policies sold by month for the top 3 states with the highest number of policies sold.

In [ ]:
# calculate total policies per state and select the top 3
top_3_states = (
    df.groupby("state")["policy"]
    .count()
    .sort_values(ascending=False)
    .head(3)
    .index
)

top_3_state_month = policies_state_month.loc[top_3_states]
top_3_state_month

## 7. Analyze the effect of marketing channels on customer response rate.

In [ ]:
# create a numeric column where Yes is 1 and all other responses are 0
df["responded_yes"] = (
    df["response"].fillna("").str.lower() == "yes"
).astype(int)

channel_response = (
    df.groupby("sales_channel")
    .agg(
        total_customers=("customer", "count"),
        yes_responses=("responded_yes", "sum"),
        response_rate=("responded_yes", "mean")
    )
    .sort_values("response_rate", ascending=False)
)

channel_response["response_rate"] = (
    channel_response["response_rate"] * 100
).round(2)

channel_response

### Conclusion
Agent has the highest response rate at 18.01%. Web, Branch and Call Center are all close to 10–11%, with Call Center lowest at 10.32%. Based on this result, Agent appears to be the most effective channel for positive campaign responses.